# US Equities — Exploratory Data Analysis

**Docker image**: `ml4t`

**Purpose**: Profile the Wiki Prices dataset of US equity OHLCV history and confirm
the inactive-symbol coverage that makes it usable for survivorship-bias-free
backtests.

**Learning objectives**:

- Load the equity panel via `data.load_us_equities` and inspect its canonical schema.
- Distinguish raw and split/dividend-adjusted price columns.
- Quantify the share of symbols that stop trading before the dataset end date.
- Check OHLC invariants and null rates across the full panel.

**Book reference**: §2.2 ("The Asset-Class Market Data Landscape" — Equities).

**Prerequisites**: `data` package on `PYTHONPATH`; Wiki Prices parquet present at
`ML4T_DATA_PATH/equities/market/us_equities/`. Run
`python data/equities/market/us_equities/download.py` if missing.

In [1]:
"""US Equities — Exploratory data analysis of the Wiki Prices dataset."""

import polars as pl

from data import load_us_equities
from utils.data_quality import check_ohlc_invariants

In [2]:
# Production defaults — Papermill injects overrides for CI
MAX_SYMBOLS = 0  # 0 = all

## 1. Load and Inspect

In [3]:
wiki = load_us_equities()

print("=== Wiki Prices Dataset ===")
print(f"Shape: {wiki.shape}")
print(f"Columns: {wiki.columns}")

=== Wiki Prices Dataset ===
Shape: (15389314, 14)
Columns: ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'ex-dividend', 'split_ratio', 'adj_open', 'adj_high', 'adj_low', 'adj_close', 'adj_volume']


In [4]:
# Schema overview
print("\nSchema:")
for col, dtype in wiki.schema.items():
    print(f"  {col}: {dtype}")


Schema:
  symbol: String
  timestamp: Date
  open: Float64
  high: Float64
  low: Float64
  close: Float64
  volume: Float64
  ex-dividend: Float64
  split_ratio: Float64
  adj_open: Float64
  adj_high: Float64
  adj_low: Float64
  adj_close: Float64
  adj_volume: Float64


### Adjusted vs Raw Prices

**Important**: This dataset contains both raw and adjusted prices.

| Column Type | Examples | Use Case |
|-------------|----------|----------|
| Raw | `open`, `high`, `low`, `close`, `volume` | Historical analysis at actual prices |
| Adjusted | `adj_open`, `adj_high`, `adj_low`, `adj_close`, `adj_volume` | **Backtesting** (handles splits/dividends) |

Always use `adj_*` columns for return calculations and strategy backtesting.

## 2. Coverage Summary

In [5]:
# Overall coverage
print("=== Coverage ===")
print(f"Unique tickers: {wiki['symbol'].n_unique():,}")
print(f"Date range: {wiki['timestamp'].min()} to {wiki['timestamp'].max()}")
print(f"Total rows: {len(wiki):,}")

=== Coverage ===
Unique tickers: 3,199
Date range: 1962-01-02 to 2018-03-27
Total rows: 15,389,314


In [6]:
# Per-stock statistics
stock_stats = wiki.group_by("symbol").agg(
    [
        pl.len().alias("days"),
        pl.col("timestamp").min().alias("start"),
        pl.col("timestamp").max().alias("end"),
        pl.col("adj_close").mean().alias("avg_price"),
    ]
)

Per-symbol coverage distribution — number of trading days and mean adjusted price.

In [7]:
stock_stats.select(["days", "avg_price"]).describe()

statistic,days,avg_price
str,f64,f64
"""count""",3199.0,3199.0
"""null_count""",0.0,0.0
"""mean""",4810.663957,118.245432
"""std""",2621.170846,2893.682465
"""min""",4.0,0.669129
"""25%""",2611.0,11.539385
"""50%""",4974.0,18.519854
"""75%""",6666.0,29.777315
"""max""",14155.0,132897.234865


## 3. Survivorship Analysis

A key feature of this dataset is that it includes stocks that ceased trading
before the dataset end date. This is critical for avoiding survivorship bias.

**Note**: Stocks marked as "inactive before end" include both:
- Actually delisted companies (bankruptcy, acquisition, etc.)
- Stocks with incomplete coverage in this dataset

The important point: these stocks are included, preventing survivorship bias.

In [8]:
dataset_end = wiki.select(pl.col("timestamp").max()).item()

# Identify stocks that stopped trading before dataset end
stock_stats = stock_stats.with_columns((pl.col("end") < dataset_end).alias("inactive_before_end"))

n_active = stock_stats.filter(~pl.col("inactive_before_end")).height
n_inactive = stock_stats.filter(pl.col("inactive_before_end")).height
total = n_active + n_inactive
inactive_pct = n_inactive / total * 100

print("=== Survivorship Analysis ===")
print(f"Dataset end: {dataset_end}")
print(f"Active at dataset end: {n_active:,} ({n_active / total * 100:.1f}%)")
print(f"Inactive before end: {n_inactive:,} ({inactive_pct:.1f}%)")
print(f"\nThis {inactive_pct:.0f}% inactive rate helps mitigate survivorship bias in backtests.")

=== Survivorship Analysis ===
Dataset end: 2018-03-27
Active at dataset end: 2,422 (75.7%)
Inactive before end: 777 (24.3%)

This 24% inactive rate helps mitigate survivorship bias in backtests.


## 4. Data Quality

In [9]:
# Check for nulls across columns
null_counts = wiki.null_count()
total_nulls = null_counts.sum_horizontal()[0]
print("=== Data Quality ===")
print(f"Total null values: {total_nulls:,}")

# Show per-column breakdown (only columns with nulls)
for col in null_counts.columns:
    val = null_counts[col][0]
    if val > 0:
        print(f"  {col}: {val:,} ({val / len(wiki) * 100:.4f}%)")

print(f"\nNull rate: {total_nulls / (len(wiki) * len(wiki.columns)) * 100:.4f}%")

=== Data Quality ===
Total null values: 1,299
  open: 538 (0.0035%)
  high: 55 (0.0004%)
  low: 55 (0.0004%)
  close: 1 (0.0000%)
  split_ratio: 1 (0.0000%)
  adj_open: 538 (0.0035%)
  adj_high: 55 (0.0004%)
  adj_low: 55 (0.0004%)
  adj_close: 1 (0.0000%)

Null rate: 0.0006%


In [10]:
# OHLC invariants on adjusted prices
invariants = check_ohlc_invariants(
    wiki,
    open_col="adj_open",
    high_col="adj_high",
    low_col="adj_low",
    close_col="adj_close",
    volume_col="adj_volume",
)

print("\nOHLC Invariants (adjusted prices):")
for row in invariants.iter_rows(named=True):
    status = "[OK]" if row["valid_pct"] >= 99.99 else "[WARN]"
    print(f"  {status} {row['check']}: {row['valid_pct']:.2f}%")


OHLC Invariants (adjusted prices):
  [OK] high_gte_low: 100.00%
  [OK] high_gte_open: 100.00%
  [OK] high_gte_close: 100.00%
  [OK] low_lte_open: 100.00%
  [OK] low_lte_close: 100.00%
  [OK] volume_non_negative: 100.00%


## 5. Example: Single Stock

In [11]:
# AAPL as example
aapl = wiki.filter(pl.col("symbol") == "AAPL").sort("timestamp")

print("=== AAPL Example ===")
print(f"Trading days: {len(aapl):,}")
print(f"Date range: {aapl['timestamp'].min()} to {aapl['timestamp'].max()}")

=== AAPL Example ===
Trading days: 9,400
Date range: 1980-12-12 to 2018-03-27


Five most recent trading days for AAPL on adjusted prices.

In [12]:
aapl.select(["timestamp", "adj_open", "adj_high", "adj_low", "adj_close", "adj_volume"]).tail(5)

timestamp,adj_open,adj_high,adj_low,adj_close,adj_volume
date,f64,f64,f64,f64,f64
2018-03-21,175.04,175.09,171.26,171.27,3.5247358e7
2018-03-22,170.0,172.68,168.6,168.845,4.1051076e7
2018-03-23,168.39,169.92,164.94,164.94,4.0248954e7
2018-03-26,168.07,173.1,166.44,172.77,3.6272617e7
2018-03-27,173.68,175.15,166.92,168.34,3.8962839e7


## Key Takeaways

1. **Mitigates survivorship bias**: 24% of symbols stop trading before the
   dataset end — including these inactive tickers is what makes the panel
   usable for unbiased backtests.
2. **Always use adjusted prices for returns**: the `adj_*` columns absorb
   splits and dividends; the raw `open/high/low/close/volume` columns remain
   available for analyses that need actual traded levels.
3. **Long history, broad cross-section**: 3,199 symbols with a max single-symbol
   span of ~56 years (14,155 trading days), covering multiple market regimes.
4. **Clean panel**: null rate is 0.0006% of values and the six adjusted-price
   OHLC invariants hold on 100% of rows.

**Next**: `02_corporate_actions` validates the adjustment factors that
produce the `adj_*` columns. **Book reference**: §2.2 (Equities).